# Python 闭包与作用域：从原理到工程实战

> 教学级 Notebook · 业务场景：LLMOps API 服务开发（模型客户端工厂 / 限流 / 缓存 / 重试 / 事件回调）

## 学习目标

完成本 Notebook 后，你将能够：

1. 用 **LEGB 规则**准确推断任意变量的解析结果；
2. 解释**闭包的本质**（函数 + cell 对象），并能用 `__closure__` / `co_freevars` 验证；
3. 正确使用 `global` 与 `nonlocal`，并能预判 `UnboundLocalError`；
4. 识别并修复**延迟绑定、循环变量共享、可变默认参数**等高频陷阱；
5. 在 LLMOps API 场景中，用闭包实现工厂、限流、缓存、重试、回调等工程模式，并能说明与「类实现」的取舍。

## 一句话本质总结

> **作用域决定变量能从哪里访问；闭包让函数能"记住"它出生时的环境——记住的是变量本身（cell），而不是当时的值。**


## 第 0 章 · 前置知识

**本章解决什么问题**：闭包建立在三个语言地基之上——命名空间、函数是一等对象、嵌套函数。先用可运行的代码把这三块地基夯实，后面的内容才有落点。

### 0.1 命名空间：变量名 → 对象的映射

- Python 中"变量"只是**名字**，通过赋值绑定到对象；名字存放在**命名空间**（本质近似字典）中。
- 模块有全局命名空间（`globals()`），每次函数调用会创建独立的局部命名空间（`locals()`）。
- 同一个名字可以在不同命名空间共存、互不干扰——这是作用域存在的根本原因。

In [1]:
# 【0.1 验证】同一名字 x 在全局与局部命名空间中各自独立
x = "全局的 x"            # ① 绑定到模块全局命名空间

def demo():
    x = "局部的 x"        # ② 绑定到本次调用的局部命名空间，不影响全局
    print("函数内 locals()['x'] =", locals()["x"])

demo()
print("函数外 globals()['x'] =", globals()["x"])
print("两次绑定互不影响：", x == "全局的 x")


函数内 locals()['x'] = 局部的 x
函数外 globals()['x'] = 全局的 x
两次绑定互不影响： True


### 0.2 函数是一等对象

函数可以像普通对象一样：被赋值给变量、作为参数传递、作为返回值返回。**闭包的全部威力都建立在第 3 点——函数可以被"带出"它出生的作用域。**

In [2]:
# 【0.2 验证】函数作为一等对象的三种用法

# ① 赋值给变量
def greet(name):
    return f"你好, {name}"

alias = greet                       # 函数对象被另一个名字引用
print("① 赋值后调用:", alias("黄桃"))

# ② 作为参数传递
def apply(func, value):
    return func(value)              # func 是传进来的函数对象

print("② 作为参数:", apply(greet, "LLMOps"))

# ③ 作为返回值返回（闭包的入口）
def make_greeter(prefix):
    def greeter(name):              # 嵌套函数
        return f"{prefix}, {name}"
    return greeter                  # 把函数对象"带出"定义它的作用域

hi = make_greeter("Hi")
print("③ 作为返回值:", hi("Agent"))
print("③ hi 的类型:", type(hi).__name__)


① 赋值后调用: 你好, 黄桃
② 作为参数: 你好, LLMOps
③ 作为返回值: Hi, Agent
③ hi 的类型: function


### 0.3 嵌套函数与自由变量

- 在另一个函数内部定义的函数叫**嵌套函数（nested function）**。
- 内层函数中"引用了、但既不是自己局部的、也不是全局/内置的"变量，称为**自由变量（free variable）**。
- 自由变量是闭包的"原材料"。可以用 `__code__.co_freevars` 直接查看一个函数有哪些自由变量。

In [3]:
# 【0.3 验证】自由变量可以被解释器明确识别
def outer():
    a, b = 1, 2                     # 外层局部变量
    def inner():
        return a                    # inner 只引用了 a，没有引用 b
    return inner

f = outer()
print("inner 的自由变量 co_freevars:", f.__code__.co_freevars)  # 只有 a
print("inner 的局部变量 co_varnames:", f.__code__.co_varnames)
print("结论：b 没有被引用，因此不会成为自由变量，也不会进入闭包")


inner 的自由变量 co_freevars: ('a',)
inner 的局部变量 co_varnames: ()
结论：b 没有被引用，因此不会成为自由变量，也不会进入闭包


## 第 1 章 · Python 作用域基础

**本章解决什么问题**：给你一个名字，Python 到底去哪里找它？掌握 LEGB 查找顺序、"哪些语法会引入新作用域"、以及"局部变量在编译期就确定"这三条规则，就能预判所有变量解析行为。

### 1.1 LEGB 规则

Python 查找变量名时，按 **L → E → G → B** 的顺序逐层向外查找，找到即止：

| 层级 | 英文 | 含义 | 引入方式 |
|---|---|---|---|
| L | Local | 当前函数内部 | 函数调用 |
| E | Enclosing | 外层嵌套函数 | 函数嵌套（可多层） |
| G | Global | 模块全局 | 模块顶层代码 |
| B | Built-in | 内置命名空间 | 解释器内置（`len`、`print` 等） |

In [4]:
# 【1.1 验证】LEGB 四层查找，逐层"撤掉"内层定义观察结果变化
x = "global(B 之外最外层)"          # G 层

def outer():
    x = "enclosing"                 # E 层
    def inner():
        x = "local"                 # L 层
        print("① L 命中:", x)
    inner()

outer()

def outer2():
    x = "enclosing"                 # E 层
    def inner():
        print("② inner 无 x → E 命中:", x)
    inner()

outer2()

def outer3():
    def inner():
        print("③ outer 也无 x → G 命中:", x)
    inner()

outer3()

# ④ B 层：连全局都没有定义的名字，最终落到内置命名空间
print("④ B 命中（内置函数 len）:", len([1, 2, 3]))


① L 命中: local
② inner 无 x → E 命中: enclosing
③ outer 也无 x → G 命中: global(B 之外最外层)
④ B 命中（内置函数 len）: 3


### 1.2 哪些语法会引入新作用域

**只有函数、类、模块会引入新作用域**。`if`、`for`、`while`、`try` 都**不会**——这与 C / Java 的块级作用域不同，是从其他语言转来时的第一大认知坑。

In [5]:
# 【1.2 验证】for / if 块内定义的变量在块外依然可见
for i in range(3):
    pass
print("for 循环结束后 i 仍在当前作用域:", i)   # i 泄漏到外层

if True:
    y = 10
print("if 块内定义的 y 在块外可见:", y)

try:
    z = "try 里定义"
except Exception:
    pass
print("try 块内定义的 z 在块外可见:", z)

def scope_check():
    return i, y, z                  # 函数内也能读到（经 G 层）
print("它们都绑定在模块全局命名空间:", scope_check())


for 循环结束后 i 仍在当前作用域: 2
if 块内定义的 y 在块外可见: 10
try 块内定义的 z 在块外可见: try 里定义
它们都绑定在模块全局命名空间: (2, 10, 'try 里定义')


### 1.3 变量绑定在编译期确定（重要）

> **规则：只要函数体内任何位置对某变量有赋值语句，该变量就在整个函数内被视为局部变量**——这是编译期（函数定义时）就确定的，与赋值语句在运行时是否执行无关。

推论：在赋值语句之前访问它，不会"向外层查找"，而是直接抛 `UnboundLocalError`。

In [6]:
# 【1.3 反例 · 先触发错误】赋值语句让 x 在整个函数内成为局部变量
x = 10

def f_wrong():
    print(x)                        # 试图读取 → 此时局部 x 尚未绑定
    x = 20                          # 正是这一行，让 x 被编译期标记为局部变量

try:
    f_wrong()
except UnboundLocalError as e:
    print("触发 UnboundLocalError:", e)

# 【1.3 正确写法】要修改全局变量，必须用 global 显式声明
def f_right():
    global x                        # 声明：本函数内的 x 指全局命名空间的 x
    x = 20

f_right()
print("global 声明后修改成功, 全局 x =", x)


触发 UnboundLocalError: cannot access local variable 'x' where it is not associated with a value
global 声明后修改成功, 全局 x = 20


## 第 2 章 · 闭包是什么

**本章解决什么问题**：外层函数都执行完了、栈帧都销毁了，内层函数凭什么还能访问外层的变量？答案是闭包——变量被"搬家"到了一个叫 cell 的容器里，随函数对象一起存活。

### 2.1 定义与成立条件

> **闭包 = 函数 + 定义该函数时的环境（自由变量）。**

成立的三个条件（缺一不可）：

1. 存在**嵌套函数**；
2. 内层函数**引用了外层函数的变量**（自由变量）；
3. 外层函数**返回内层函数**（或内层函数被传递到别处，使其生命周期超过外层调用）。

In [7]:
# 【2.1 验证】外层函数已结束，内层函数仍能访问外层变量
def outer(msg):
    def inner():
        print("闭包内读取 msg:", msg)   # msg 是 inner 的自由变量
    return inner

f = outer("hello")                  # outer 调用结束，其栈帧已销毁
f()                                 # 但 msg 依然可用 —— 它被保存在闭包的 cell 中

# 查看闭包的底层结构
print("f.__closure__ 长度:", len(f.__closure__))
print("第 0 个 cell 的内容:", f.__closure__[0].cell_contents)
print("自由变量名:", f.__code__.co_freevars)


闭包内读取 msg: hello
f.__closure__ 长度: 1
第 0 个 cell 的内容: hello
自由变量名: ('msg',)


### 2.2 闭包保存的是"变量"，不是"值"（核心认知）

闭包捕获的是**变量容器（cell）的引用**，所有共享该变量的闭包看到的是**同一个变量的当前值**。这是延迟绑定陷阱的总根源。

In [8]:
# 【2.2 反例 · 先触发错误】循环中创建的 lambda 共享同一个 i
def make_funcs():
    funcs = []
    for i in range(3):
        funcs.append(lambda: i)     # 三个 lambda 捕获的是同一个变量 i
    return funcs

fs = make_funcs()
print("错误写法输出:", [f() for f in fs])   # 循环结束时 i=2，全部看到 2

# 【2.2 修复一】用默认参数在"定义时"立即绑定当前值
def make_funcs_fix1():
    funcs = []
    for i in range(3):
        funcs.append(lambda i=i: i)  # i=i 把当前值拷贝进各自的默认参数
    return funcs

print("修复一（默认参数）:", [f() for f in make_funcs_fix1()])

# 【2.2 修复二】用工厂函数给每个 i 创建独立的作用域
def make_func(i):
    return lambda: i                 # 每次调用 make_func 都产生独立的 cell

funcs = [make_func(i) for i in range(3)]
print("修复二（工厂函数）:", [f() for f in funcs])


错误写法输出: [2, 2, 2]
修复一（默认参数）: [0, 1, 2]
修复二（工厂函数）: [0, 1, 2]


## 第 3 章 · nonlocal 与 global

**本章解决什么问题**：闭包默认只能"读"外层变量，想"改"必须显式声明改的是哪一层。`global` 与 `nonlocal` 就是两个"层级声明器"。

| 关键字 | 声明变量属于 | 典型用途 | 不能做的事 |
|---|---|---|---|
| `global` | 模块全局作用域 | 修改模块级状态 | 指向外层函数变量 |
| `nonlocal` | 最近的外层函数作用域 | 修改闭包捕获的变量 | 指向全局变量；跨越模块边界 |

In [9]:
# 【3.1 验证 global】修改模块级变量
count = 0

def inc():
    global count                    # 声明指向 G 层
    count += 1

inc(); inc()
print("global 修改模块变量 count =", count)

# 【3.2 验证 nonlocal】修改外层函数变量 —— 闭包计数器
def counter():
    count = 0                       # 外层函数变量，将被 cell 化
    def inc():
        nonlocal count              # 声明指向最近的外层函数作用域
        count += 1
        return count
    return inc

c = counter()
print("nonlocal 计数器:", c(), c(), c())


global 修改模块变量 count = 2
nonlocal 计数器: 1 2 3


In [10]:
# 【3.3 反例 · 先触发错误】没有 nonlocal 时，+= 触发 UnboundLocalError
def counter_wrong():
    count = 0
    def inc():
        count += 1                  # 等价于 count = count + 1 → 含赋值 → count 被标记为 inc 的局部变量
        return count
    return inc

try:
    counter_wrong()()
except UnboundLocalError as e:
    print("触发 UnboundLocalError:", e)

# 【3.3 对比】加上 nonlocal 即修复（见上一单元），此处验证另一种思路：用可变对象绕开重新绑定
def counter_mutable():
    count = [0]                     # 用列表承载状态
    def inc():
        count[0] += 1               # 修改对象内容 ≠ 重新绑定变量名，无需 nonlocal
        return count[0]
    return inc

cm = counter_mutable()
print("可变对象方案:", cm(), cm(), cm())
print("规律：重新绑定名字需要 nonlocal；原地修改可变对象不需要")


触发 UnboundLocalError: cannot access local variable 'count' where it is not associated with a value
可变对象方案: 1 2 3
规律：重新绑定名字需要 nonlocal；原地修改可变对象不需要


## 第 4 章 · 闭包的本质：cell 对象

**本章解决什么问题**：外层函数的栈帧销毁后，变量到底"活"在哪里？答案：解释器在编译期发现某变量被嵌套函数引用时，会把它从栈上的普通局部变量升级为堆上的 **cell 对象**；外层、内层函数都通过 cell 间接读写它。栈帧没了，cell 还在，被内层函数的 `__closure__` 元组引用着——这解释了闭包"记住的是变量而不是值"。

In [11]:
# 【4.1 验证】通过 cell_contents 直接读写 cell —— 证明闭包持有的是"活"的容器
def outer():
    x = 10
    def inner():
        return x
    return inner

f = outer()
cell = f.__closure__[0]
print("读取 cell 内容:", cell.cell_contents)

cell.cell_contents = 99             # 直接改写 cell
print("改写 cell 后调用 f():", f())  # 闭包读到的是新值 → 证明存的是容器而非快照

# 【4.2 验证】只有被内层函数引用的变量才会 cell 化
def outer2():
    a = 1
    b = 2                           # b 没有被 inner 引用
    def inner():
        return a
    return inner

f2 = outer2()
print("co_freevars:", f2.__code__.co_freevars, "→ b 不在闭包中, 不占用 cell")


读取 cell 内容: 10
改写 cell 后调用 f(): 99
co_freevars: ('a',) → b 不在闭包中, 不占用 cell


## 第 5 章 · 闭包的常见使用场景（概览）

**本章解决什么问题**：闭包不是语法糖摆设，它是"轻量级状态封装"的通用武器。先快速过一遍 7 大场景的最小骨架，第 8 章再给出面向 LLMOps 业务的完整工程实现。

| 场景 | 闭包扮演的角色 |
|---|---|
| 工厂函数 | 把配置固化进函数 |
| 计数器 | 私有状态，外部不可篡改 |
| 装饰器 | 包裹被装饰函数，注入横切逻辑 |
| 回调 / 事件处理 | 把上下文带进回调 |
| 配置化函数 | 参数先行、逻辑后置 |
| 缓存 / 记忆化 | 闭包私有 cache |
| 偏函数模拟 | 预填部分参数 |

In [12]:
# 【5.1 工厂函数】把指数配置固化进函数
def power(exp):
    def calc(base):
        return base ** exp
    return calc

square, cube = power(2), power(3)
print("工厂函数: square(5) =", square(5), ", cube(5) =", cube(5))

# 【5.2 独立计数器】每个闭包实例有独立环境
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c1, c2 = make_counter(), make_counter()
print("c1:", c1(), c1(), "| c2:", c2(), "→ 状态相互独立")


工厂函数: square(5) = 25 , cube(5) = 125
c1: 1 2 | c2: 1 → 状态相互独立


In [13]:
# 【5.3 装饰器】装饰器的本质就是闭包
from functools import wraps

def log(func):
    @wraps(func)                    # 保留原函数的 __name__ 等元信息
    def wrapper(*args, **kwargs):   # wrapper 闭包捕获了 func
        print(f"[log] 调用 {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log
def add(a, b):
    return a + b

print("add(1, 2) =", add(1, 2))
print("函数名被 wraps 保留:", add.__name__)

# 【5.4 回调】把名字（上下文）带进事件处理器
def make_handler(name):
    def handler(event):
        print(f"{name} 处理事件: {event}")
    return handler

btn1, btn2 = make_handler("按钮1"), make_handler("按钮2")
btn1("click"); btn2("hover")


[log] 调用 add
add(1, 2) = 3
函数名被 wraps 保留: add
按钮1 处理事件: click
按钮2 处理事件: hover


## 第 6 章 · 闭包与类的对比

**本章解决什么问题**：闭包和类都能"携带状态"，工程上什么时候用哪个？

| 维度 | 闭包 | 类 |
|---|---|---|
| 代码量 | 少 | 多 |
| 状态可见性 | 隐蔽（私有，不易查看） | 显式（实例属性，可查看） |
| 多方法 | 不方便（需返回多个函数或字典） | 方便 |
| 继承 / 多态 | 不支持 | 支持 |
| 调试 | 较难（状态藏在 cell 里） | 容易 |
| 适用 | 单一行为 + 少量状态 | 复杂对象、多行为、需扩展 |

> **经验法则**：只需要"一个会记事的函数"时用闭包；需要"一个对象"时用类。

In [14]:
# 【6.1 对比】同一个计数器的两种实现
# 闭包版
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

# 类版（实现 __call__ 让实例可像函数一样调用）
class Counter:
    def __init__(self):
        self.count = 0
    def __call__(self):
        self.count += 1
        return self.count

f, o = make_counter(), Counter()
print("闭包版:", f(), f(), f())
print("类版:  ", o(), o(), o())
print("类版状态可直接检查: o.count =", o.count, "（闭包的状态则藏在 cell 中）")


闭包版: 1 2 3
类版:   1 2 3
类版状态可直接检查: o.count = 3 （闭包的状态则藏在 cell 中）


In [15]:
# 【6.2 修正说明】源文档称"闭包性能通常更快"——这不是可靠结论，应以测量为准。
# 确定性的证据：两者读取状态的字节码路径不同（LOAD_DEREF vs LOAD_ATTR），
# 在现代 CPython 中两者都是高度优化的操作，差异通常在纳秒级，不构成选型依据。
import dis

def closure_counter():
    count = 0
    def inc():
        nonlocal count
        count += 1
        return count
    return inc

class ClassCounter:
    def __init__(self):
        self.count = 0
    def inc(self):
        self.count += 1
        return self.count

c_inc = closure_counter()
print("闭包版读取状态的字节码:", [i.opname for i in dis.get_instructions(c_inc) if "LOAD" in i.opname])
print("类版读取状态的字节码:  ", [i.opname for i in dis.get_instructions(ClassCounter.inc) if "LOAD" in i.opname])
print("结论：选型应依据状态可见性/扩展性，而非假定闭包更快")


闭包版读取状态的字节码: ['LOAD_DEREF', 'LOAD_CONST', 'LOAD_DEREF']
类版读取状态的字节码:   ['LOAD_FAST', 'LOAD_ATTR', 'LOAD_CONST', 'LOAD_FAST', 'LOAD_ATTR']
结论：选型应依据状态可见性/扩展性，而非假定闭包更快


## 第 7 章 · 作用域的特殊情况

**本章解决什么问题**：LEGB 有四条例外规则——类作用域、推导式作用域、默认参数求值时机、cell 化条件。它们不常遇到，但每一条都对应一类"灵异 bug"。

In [16]:
# 【7.1 验证】类作用域不参与闭包：方法内的名字查找会"跳过"类体
x = "global"

class A:
    x = "class"                     # 类体是独立作用域，但不构成 E 层
    def method(self):
        print("方法内读到 x =", x)  # 跳过类作用域，直接命中 G 层

A().method()
print("访问类属性需要显式限定: A.x =", A.x)


方法内读到 x = global
访问类属性需要显式限定: A.x = class


In [17]:
# 【7.2 验证】推导式有自己的作用域（Python 3），普通 for 循环没有
x = 10
lst = [x for x in range(3)]         # 推导式内的 x 是推导式私有的
print("推导式之后 x 不受影响:", x)

for x in range(3):                  # 普通 for 不引入作用域，x 泄漏到外面
    pass
print("for 循环之后 x 被改写:", x)

# 同理，生成器表达式/字典推导也有自己的作用域
gen = (i * 2 for i in range(3))
print("生成器表达式独立作用域:", list(gen))


推导式之后 x 不受影响: 10
for 循环之后 x 被改写: 2
生成器表达式独立作用域: [0, 2, 4]


In [18]:
# 【7.3 反例 · 先触发错误】默认参数在"定义时"只求值一次 —— 可变默认参数陷阱
def f_wrong(x=[]):                  # 这个 [] 只创建一次，被所有调用共享
    x.append(1)
    return x

print("错误写法三次调用:", f_wrong(), f_wrong(), f_wrong())  # 列表越积越长

# 【7.3 正确写法】用 None 做哨兵，函数体内再创建新对象
def f_right(x=None):
    if x is None:
        x = []                      # 每次调用都新建列表
    x.append(1)
    return x

print("正确写法三次调用:", f_right(), f_right(), f_right())


错误写法三次调用: [1, 1, 1] [1, 1, 1] [1, 1, 1]
正确写法三次调用: [1] [1] [1]


## 第 8 章 · 实用示例集（面向 LLMOps API 业务）

**本章解决什么问题**：把闭包模式落到 LLMOps API 服务的高频需求上。以下 5 个模式全部可运行、输出确定（时钟与"模型调用"均用可注入的确定性替身）。

| # | 模式 | LLMOps 场景 | 注意点 |
|---|---|---|---|
| 1 | 工厂函数 | 按配置生成模型客户端调用器 | 配置即闭包环境，改配置=换新实例 |
| 2 | 限流器 | API 网关调用频次限制 | 用可变对象+闭包共享状态；时钟可注入 |
| 3 | 记忆化缓存 | 相同 prompt 命中缓存，省 token | cache 私有；注意 key 必须可哈希 |
| 4 | 配置化重试 | 对不稳定的模型调用自动重试 | 只重试可恢复异常；最后一次要抛出 |
| 5 | 事件回调 | 流式输出的 token 回调 / 事件总线 | 回调注册顺序即触发顺序 |

In [19]:
# 【8.1 模式一：模型客户端工厂】把"服务商配置"固化进调用函数
def make_llm_client(provider, model, temperature):
    """工厂：每次调用生成一个绑定特定配置的 chat 函数（闭包捕获三个配置项）。"""
    def chat(prompt):                               # chat 是闭包，捕获 provider/model/temperature
        return f"[{provider}/{model}@temp={temperature}] 回复: {prompt}..."
    return chat

gpt = make_llm_client("openai", "gpt-4o", 0.7)
claude = make_llm_client("anthropic", "claude-sonnet", 0.3)

print(gpt("解释 RAG"))
print(claude("解释 RAG"))
print("自由变量:", gpt.__code__.co_freevars)


[openai/gpt-4o@temp=0.7] 回复: 解释 RAG...
[anthropic/claude-sonnet@temp=0.3] 回复: 解释 RAG...
自由变量: ('model', 'provider', 'temperature')


In [20]:
# 【8.2 模式二：限流器】滑动窗口限流 —— 时钟注入保证输出可复现
def rate_limiter(max_calls, period, clock):
    """返回一个 check() 函数；calls 列表被闭包持有，在多次调用间共享。
    clock 由外部注入（测试时给假时钟），保证行为确定性。"""
    calls = []                                      # 闭包私有状态：时间戳窗口
    def check():
        now = clock()
        calls[:] = [t for t in calls if now - t < period]  # 原地清理过期记录（无需 nonlocal）
        if len(calls) >= max_calls:
            return "REJECT"
        calls.append(now)
        return "OK"
    return check

# 假时钟：手动推进时间，输出完全确定
class FakeClock:
    def __init__(self):
        self.t = 0.0
    def __call__(self):
        return self.t

clock = FakeClock()
check = rate_limiter(max_calls=3, period=10, clock=clock)
print("同一时刻连续 5 次调用:", [check() for _ in range(5)])  # 3 次 OK 后被拒
clock.t = 11.0                                      # 时间推进，滑出窗口
print("窗口滑过后再调用:", check())


同一时刻连续 5 次调用: ['OK', 'OK', 'OK', 'REJECT', 'REJECT']
窗口滑过后再调用: OK


In [21]:
# 【8.3 模式三：记忆化缓存】相同 prompt 不重复调用模型
def memoize(func):
    cache = {}                                      # 闭包私有缓存
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)               # 未命中才调用真实函数
        return cache[args]
    return wrapper

# 用计数器模拟"昂贵的 LLM 调用"，验证缓存确实拦截了重复请求
call_stats = {"n": 0}

@memoize
def fake_llm(prompt):
    call_stats["n"] += 1
    return f"<回答: {prompt}>"

print(fake_llm("什么是闭包"))
print(fake_llm("什么是闭包"))                        # 命中缓存
print(fake_llm("什么是作用域"))
print("真实模型调用次数 =", call_stats["n"], "（3 次请求只触发 2 次真实调用）")


<回答: 什么是闭包>
<回答: 什么是闭包>
<回答: 什么是作用域>
真实模型调用次数 = 2 （3 次请求只触发 2 次真实调用）


In [22]:
# 【8.4 模式四：配置化重试装饰器】对不稳定的模型调用自动重试
from functools import wraps

def with_retry(times):
    """三层嵌套：配置层(times) → 装饰器层(func) → 运行时层(args)。"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if i == times - 1:              # 最后一次失败要向上抛
                        raise
                    print(f"  第 {i + 1} 次失败, 重试中...")
        return wrapper
    return decorator

# 确定性"不稳定函数"：前 2 次必失败，第 3 次成功（用闭包计数模拟）
def make_flaky(fail_times):
    n = {"count": 0}
    def call():
        n["count"] += 1
        if n["count"] <= fail_times:
            raise RuntimeError("模型超时")
        return "调用成功"
    return call

@with_retry(3)
def unstable():
    return make_flaky(2)()                          # 每次 wrapper 调用都执行这句

# 上面的写法每次都会新建 flaky 函数，无法演示跨重试共享计数；
# 正确演示：flaky 在装饰器外部创建，闭包状态跨重试共享
flaky = make_flaky(2)

@with_retry(3)
def unstable2():
    return flaky()

print(unstable2())


  第 1 次失败, 重试中...
  第 2 次失败, 重试中...
调用成功


In [23]:
# 【8.5 模式五：事件回调总线】流式输出 / 插件钩子的基础件
def make_event_bus():
    listeners = []                                  # 闭包私有监听列表
    def on(event, handler):
        listeners.append((event, handler))          # 注册回调
    def emit(event, data):
        for e, h in listeners:
            if e == event:
                h(data)                             # 按注册顺序触发
    return on, emit

on, emit = make_event_bus()
on("token", lambda d: print("流式 token:", d))
on("done",  lambda d: print("生成完成:", d))

emit("token", "你好")
emit("token", "世界")
emit("done", "共 2 个 token")


流式 token: 你好
流式 token: 世界
生成完成: 共 2 个 token


## 第 9 章 · 内置 / 生态能力对照

**本章解决什么问题**：手写闭包之前，先确认标准库是否已有轮子。工程原则：**标准库能覆盖的，不重复造；需要定制行为的，才手写闭包。**

| 需求 | 手写闭包 | 标准库方案 | 选型建议 |
|---|---|---|---|
| 预填部分参数 | `partial(func, *fixed)` 自实现 | `functools.partial` | 一律用标准库 |
| 记忆化缓存 | `@memoize` 自实现 | `functools.lru_cache(maxsize=...)` | 标准库带容量淘汰，优先用 |
| 计数器 | `nonlocal count` | `itertools.count` | 简单递增用标准库 |
| 保留函数元信息 | 手动拷贝 `__name__` 等 | `functools.wraps` | 写装饰器必用 |
| 限流 / 重试 | 手写闭包 | 三方库 `limits` / `tenacity` | 生产环境建议成熟库 |

In [24]:
# 【9.1 对照】手写偏函数 vs functools.partial
def my_partial(func, *fixed_args):
    def wrapper(*args, **kwargs):
        return func(*fixed_args, *args, **kwargs)   # 闭包捕获 fixed_args
    return wrapper

def greet(greeting, name):
    return f"{greeting}, {name}"

from functools import partial
a = my_partial(greet, "Hello")
b = partial(greet, "Hello")
print("手写闭包:", a("Alice"))
print("标准库:  ", b("Alice"), "→ 输出一致")

# 【9.2 对照】手写 memoize vs functools.lru_cache
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("lru_cache 版 fib(50) =", fib(50))
print("缓存信息:", fib.cache_info())


手写闭包: Hello, Alice
标准库:   Hello, Alice → 输出一致
lru_cache 版 fib(50) = 12586269025
缓存信息: CacheInfo(hits=48, misses=51, maxsize=None, currsize=51)


In [25]:
# 【9.3 对照】nonlocal 计数器 vs itertools.count
from itertools import count as it_count

def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c = make_counter()
print("闭包计数器:", c(), c(), c())

cnt = it_count(start=1)
print("itertools :", next(cnt), next(cnt), next(cnt))


闭包计数器: 1 2 3
itertools : 1 2 3


## 第 10 章 · 综合实战：LLM API 网关调用链

**本章解决什么问题**：把前面所有知识点组合成一个贴近 LLMOps 业务的完整例子——一个极简 LLM 网关调用链，串起 **工厂 + 缓存 + 限流 + 重试 + 事件回调 + 计数器** 六种闭包模式。

**设计取舍说明**：

1. **时钟与模型注入**：`clock` 和 `backend` 都由外部注入，测试可复现、生产可替换真实实现；
2. **缓存放在限流之前**：命中缓存的请求不消耗限流配额（省 token 的请求不应被惩罚）；
3. **重试只包裹真实调用**：缓存命中走的是快路径，无需重试；
4. **状态全部私有化**：`cache`、`calls`、`stats` 都藏在各自闭包里，网关外部无法篡改；
5. **多返回值用字典组装**：一个闭包携带多个方法时不便，返回 dict 是"闭包版多方法"的常见折中（再多就该上类了）。

In [26]:
# 【10.1 综合实战】极简 LLM 网关（全部确定性，可重复执行）
from functools import wraps

# ---- 确定性基础设施：假时钟 + 假模型后端 ----
class FakeClock:
    def __init__(self): self.t = 0.0
    def __call__(self): return self.t
    def advance(self, dt): self.t += dt

def make_backend(fail_first=0):
    """假模型后端：前 fail_first 次抛异常，之后正常返回；闭包记录调用次数。"""
    stats = {"calls": 0}
    def call(prompt):
        stats["calls"] += 1
        if stats["calls"] <= fail_first:
            raise RuntimeError("后端超时")
        return f"<{prompt} 的回答>"
    call.stats = stats
    return call

# ---- 网关组装：闭包模式的组合 ----
def make_gateway(backend, max_calls=3, period=10, retry=2, clock=None):
    clock = clock or FakeClock()
    cache, window = {}, []                          # 闭包私有状态：缓存 + 限流窗口
    on_event, handlers = (lambda e, h: handlers.setdefault(e, []).append(h)), {}
    def emit(event, data):
        for h in handlers.get(event, []):
            h(data)

    def retrying_call(prompt):                      # 重试只包真实调用
        for i in range(retry + 1):
            try:
                return backend(prompt)
            except RuntimeError:
                if i == retry:
                    raise
                emit("retry", f"第 {i + 1} 次重试: {prompt}")

    def chat(prompt):                               # 网关主入口
        if prompt in cache:                         # ① 缓存优先，不耗限流配额
            emit("cache_hit", prompt)
            return cache[prompt]
        now = clock()
        window[:] = [t for t in window if now - t < period]  # ② 滑动窗口限流
        if len(window) >= max_calls:
            emit("reject", prompt)
            return "REJECT: 调用过于频繁"
        window.append(now)
        result = retrying_call(prompt)              # ③ 带重试的真实调用
        cache[prompt] = result                      # ④ 写入缓存
        return result

    return {"chat": chat, "on": on_event, "emit": emit,
            "cache_size": lambda: len(cache)}

# ---- 运行演示 ----
clock = FakeClock()
backend = make_backend(fail_first=1)                # 首次调用必超时
gw = make_gateway(backend, max_calls=3, period=10, retry=2, clock=clock)
gw["on"]("retry", lambda d: print("  [事件]", d))
gw["on"]("cache_hit", lambda d: print("  [事件] 缓存命中:", d))
gw["on"]("reject", lambda d: print("  [事件] 限流拒绝:", d))

print("1)", gw["chat"]("RAG 是什么"))               # 第1次：后端超时→重试→成功
print("2)", gw["chat"]("RAG 是什么"))               # 第2次：缓存命中
print("3)", gw["chat"]("什么是闭包"))
print("4)", gw["chat"]("什么是装饰器"))
print("5)", gw["chat"]("什么是生成器"))             # 窗口已满 → 限流
clock.advance(11)                                   # 时间推进，窗口滑出
print("6)", gw["chat"]("什么是生成器"))             # 恢复可用
print("后端真实调用次数:", backend.stats["calls"], "| 缓存条目:", gw["cache_size"]())


  [事件] 第 1 次重试: RAG 是什么
1) <RAG 是什么 的回答>
  [事件] 缓存命中: RAG 是什么
2) <RAG 是什么 的回答>
3) <什么是闭包 的回答>
4) <什么是装饰器 的回答>
  [事件] 限流拒绝: 什么是生成器
5) REJECT: 调用过于频繁
6) <什么是生成器 的回答>
后端真实调用次数: 5 | 缓存条目: 4


## 第 11 章 · 常见陷阱汇总

**本章解决什么问题**：把分散在各章的陷阱集中成一张"排障索引"，并附最小复现代码。

| 陷阱 | 症状 | 根因 | 解法 |
|---|---|---|---|
| 函数内"先读后写" | `UnboundLocalError` | 赋值让变量在编译期成为局部变量 | `global` / `nonlocal` 显式声明 |
| 延迟绑定 | 循环生成的闭包全部返回最后的值 | 闭包共享同一个循环变量 | 默认参数 `i=i` 或工厂函数 |
| 循环变量共享（回调版） | 所有 handler 打印同一个 name | 同上 | `lambda name=name: ...` |
| 闭包改外层不可变变量 | `UnboundLocalError` | `+=` 属于重新绑定 | `nonlocal` 或改用可变对象 |
| 可变默认参数 | 多次调用结果"越积越多" | 默认参数定义时只求值一次 | 用 `None` 哨兵 |
| 闭包持有大对象 | 内存不释放 | cell 长生命周期持有引用 | 避免捕获大对象，或 `del` 释放 |

In [27]:
# 【11.1 陷阱复现一】延迟绑定 & 循环变量共享 —— 一处代码两个坑
handlers = []
for name in ["a", "b", "c"]:
    handlers.append(lambda: print("错误写法:", name))

for h in handlers:
    h()                                             # 全部打印 c

handlers = []
for name in ["a", "b", "c"]:
    handlers.append(lambda name=name: print("正确写法:", name))

for h in handlers:
    h()                                             # a / b / c


错误写法: c
错误写法: c
错误写法: c
正确写法: a
正确写法: b
正确写法: c


In [28]:
# 【11.2 陷阱复现二】闭包变量共享的"双刃剑"：可变对象跨调用累积
def outer():
    x = []
    def add(v):
        x.append(v)                                 # 修改对象本身，无需 nonlocal
        return x
    return add

add = outer()
print("第一次:", add(1))
print("第二次:", add(2))                            # [1, 2] —— 状态在累积，注意是否符合预期

# 【11.3 陷阱复现三】闭包持有大对象导致内存无法释放（原理演示）
import sys
def make_closure():
    big = [0] * 1000                                # 被闭包捕获的对象
    def use():
        return len(big)
    return use

u = make_closure()
cell = u.__closure__[0]
print("闭包持有的列表长度:", len(cell.cell_contents))
print("只要 u 存活，该列表就无法被 GC；不再需要时应 del u 释放")
del u
print("已 del u，闭包与 cell 一并释放")


第一次: [1]
第二次: [1, 2]
闭包持有的列表长度: 1000
只要 u 存活，该列表就无法被 GC；不再需要时应 del u 释放
已 del u，闭包与 cell 一并释放


## 第 12 章 · 总结

### 12.1 通用代码模板（可直接复制）

In [29]:
# 【模板一】闭包工厂：固化配置 + 私有状态
def make_service(config_value):
    state = {"count": 0}                # 私有状态（可变对象，免 nonlocal）
    def service(arg):
        state["count"] += 1
        return f"config={config_value}, arg={arg}, 第{state['count']}次"
    return service

svc = make_service("prod")
print(svc("req-1"))
print(svc("req-2"))

# 【模板二】带配置的装饰器（三层闭包标准骨架）
from functools import wraps

def configurable_decorator(option):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # 前处理（可用 option）
            result = func(*args, **kwargs)
            # 后处理
            return result
        return wrapper
    return decorator

@configurable_decorator(option="demo")
def hello():
    return "hello"

print(hello(), "| 元信息保留:", hello.__name__)


config=prod, arg=req-1, 第1次
config=prod, arg=req-2, 第2次
hello | 元信息保留: hello


### 12.2 记忆口诀

1. **LEGB 四层查，找到即止不回头**——Local → Enclosing → Global → Built-in。
2. **函数类模块，才有新作用域**——`if`/`for`/`while`/`try` 都不设防。
3. **函数内一赋值，全函数算局部**——绑定在编译期定死，先读后写必报 `UnboundLocalError`。
4. **改全局用 global，改外层用 nonlocal，改可变对象啥都不用**。
5. **闭包记的是变量不是值**——循环里建闭包，默认参数或工厂来救场。
6. **简单状态用闭包，复杂对象用类**——标准库有的轮子不重复造。

### 12.3 自测练习

1. **自由变量识别**：写出函数 `def outer(): a=1; b=2; def inner(): return a+b if a else b; return inner` 返回值的 `co_freevars` 是什么？
2. **修复延迟绑定**：`funcs = [lambda: i * i for i in range(4)]` 输出是什么？用两种方式修复为 `[0, 1, 4, 9]`。
3. **手写一个"调用次数限制器"**：`limited(3)(func)` 返回的函数最多可调用 3 次，超过返回 `"LIMIT"`（不得用类）。
4. **LEGB 推理**：全局 `x=1`；`outer` 内 `x=2`；`inner` 内 `print(x)` 且 `inner` 里还有一句 `x=3`——运行结果是什么，为什么？

In [30]:
# 【自测参考实现】先自己作答，再运行本单元核对

# ---- 练习 1 ----
def outer():
    a, b = 1, 2
    def inner():
        return a + b if a else b      # a、b 都被引用
    return inner
print("练习1:", outer().__code__.co_freevars)

# ---- 练习 2 ----
funcs = [lambda: i * i for i in range(4)]
print("练习2 错误输出:", [f() for f in funcs])
fix1 = [lambda i=i: i * i for i in range(4)]                 # 默认参数法
def make_sq(i): return lambda: i * i                          # 工厂函数法
fix2 = [make_sq(i) for i in range(4)]
print("练习2 修复:", [f() for f in fix1], [f() for f in fix2])

# ---- 练习 3 ----
def limited(n):
    def decorator(func):
        left = {"n": n}               # 可变对象承载剩余次数
        def wrapper(*args, **kwargs):
            if left["n"] <= 0:
                return "LIMIT"
            left["n"] -= 1
            return func(*args, **kwargs)
        return wrapper
    return decorator

@limited(3)
def ping():
    return "pong"

print("练习3:", [ping() for _ in range(5)])

# ---- 练习 4 ----
x = 1
def outer4():
    x = 2
    def inner():
        print(x)                      # 编译期：x 是 inner 的局部变量（因下方 x=3）
        x = 3
    return inner
try:
    outer4()()
except UnboundLocalError as e:
    print("练习4: UnboundLocalError →", e)
print("原因：inner 内存在赋值 x=3，x 在整个 inner 内是局部变量，print 时尚未绑定")


练习1: ('a', 'b')
练习2 错误输出: [9, 9, 9, 9]
练习2 修复: [0, 1, 4, 9] [0, 1, 4, 9]
练习3: ['pong', 'pong', 'pong', 'LIMIT', 'LIMIT']
练习4: UnboundLocalError → cannot access local variable 'x' where it is not associated with a value
原因：inner 内存在赋值 x=3，x 在整个 inner 内是局部变量，print 时尚未绑定
